In [ ]:
# Colab 請選 2026.07；安裝到獨立環境，避免更改核心已載入的 NumPy。
from pathlib import Path
import subprocess
import sys
import tempfile

assert sys.version_info[:3] == (3, 12, 13), "請使用書籍或 Colab 2026.07 的 Python 3.12.13"
python = sys.executable
if "google.colab" in sys.modules:
    %pip -q install virtualenv
    env = Path("/content/ch00-book-env")
    if not (env / "bin/python").is_file():
        subprocess.run([python, "-m", "virtualenv", "-q", str(env)], check=True)
    python = str(env / "bin/python")
    pins = ["deepchem==2.8.1.dev20260821122258", "numpy==1.26.4", "scipy==1.17.1",
            "rdkit==2026.9.1", "scikit-learn==1.9.1", "pandas==3.0.6", "matplotlib==3.11.2"]
    subprocess.run([python, "-m", "pip", "install", "-q", *pins], check=True)
    subprocess.run([python, "-m", "pip", "check"], check=True)
work = tempfile.TemporaryDirectory()
script = Path(work.name) / "00-01-what-is-cheminformatics.py"

# 什麼是化學資訊學

先從上到下執行。下格將分析程式寫入檔案，最後一格才使用安裝好的環境執行它。

資料為 DeepChem 載入的 ESOL，保留全部紀錄，不訓練模型、不生成假量測值。原始研究：[Delaney 2004](https://doi.org/10.1021/ci034243x)；資料彙整：[MoleculeNet](https://doi.org/10.1039/C7SC02664A)。

Colab 使用 CPU 即可。若看不到 2026.07，請先核對書籍環境說明，勿忽略版本檢查。

<!-- concept-ch00-05:start -->
把同一個分子的資訊攤開來看，就能分清楚哪一欄可以由結構計算，哪一欄必須另找量測。

| 資訊 | 乙醇與二甲醚的例子 | 能回答的問題 |
|---|---|---|
| 分子式 | 兩者都是 C₂H₆O | 元素總數相同嗎？ |
| 連接方式 | 乙醇 C–C–O；二甲醚 C–O–C | 氧接在哪裡？ |
| 莫耳質量 | 相同元素組成給出相同的計算值 | 一莫耳有多重？ |
| 實測水溶解度 | 各自保留量測值、條件與來源 | 特定條件下能溶多少？ |

由上往下讀，前兩列說明為何只有分子式還不夠；最後一列則提醒我們，結構相同也不能省略量測條件。

<!-- concept-ch00-05:end -->


In [ ]:
%%writefile "{script}"
from pathlib import Path
import json
import numpy as np
import pandas as pd
import os
# 使用適合輸出圖檔的繪圖後端，避免沿用 Colab 的顯示設定。
os.environ["MPLBACKEND"] = "Agg"
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw
import deepchem as dc

# ESOL 真實水溶解度；原始研究 DOI:10.1021/ci034243x。
# 本章只讀取、核對與篩選，不訓練模型，因此不切分資料；不轉換標籤。
tasks, parts, transformers = dc.molnet.load_delaney(
    featurizer=dc.feat.RawFeaturizer(smiles=True), splitter=None,
    transformers=[], reload=False)
data = parts[0]
assert len(tasks) == 1 and not transformers
mols = [Chem.MolFromSmiles(str(s)) for s in data.ids]
table = pd.DataFrame({"row_id": np.arange(len(data)), "smiles": data.ids,
    "logS": data.y[:, 0], "weight": data.w[:, 0],
    "parse_ok": [m is not None for m in mols]})
table["canonical_smiles"] = [Chem.MolToSmiles(m) if m is not None else None for m in mols]
table["MW"] = [Descriptors.MolWt(m) if m is not None else np.nan for m in mols]
book = Path("../..").resolve() if Path("../../_quarto.yml").is_file() else Path.cwd()
figures = book / "figures"
figures.mkdir(exist_ok=True)
outputs = Path("outputs")
outputs.mkdir(exist_ok=True)
print("Python / RDKit:", __import__("sys").version.split()[0], __import__("rdkit").__version__)
print("Task:", tasks[0], "rows:", len(table))
sample = table.head(3)
print(sample[["row_id", "smiles", "logS", "MW"]].to_string(index=False))
image = Draw.MolsToGridImage(mols[:3], molsPerRow=3, subImgSize=(360,260),
    legends=[f"row {row['row_id']} | logS {row['logS']:.2f}"
             for _, row in sample.iterrows()])
image.save(figures / "00-01-records.png")


In [ ]:
from IPython.display import Image, display
run = subprocess.run([python, str(script)], capture_output=True, text=True)
if run.returncode:
    print(run.stderr)
    run.check_returncode()
print(run.stdout.replace("This module requires PyTorch to be installed.\n", "").strip())
figure_dir = Path("../../figures") if Path("../../_quarto.yml").is_file() else Path("figures")
display(Image(filename=str(figure_dir / "00-01-records.png")))